In [1]:
from pathlib import Path
import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DB_PATH = PROJECT_ROOT / "data" / "ltv.duckdb"
BRONZE_TABLE = "bronze"   

In [2]:
con = duckdb.connect(str(DB_PATH))
con.sql("SHOW TABLES").show()

bronze = con.sql(f"SELECT * FROM {BRONZE_TABLE}").df()
print(f"Bronze rows: {len(bronze):,}   columns: {len(bronze.columns)}")

##Add more summary stuff, stats, what the column names are, 

┌─────────┐
│  name   │
│ varchar │
├─────────┤
│ bronze  │
│ silver  │
└─────────┘

Bronze rows: 1,067,371   columns: 8


In [3]:
def clean_text(value):
    """Strip spaces and lowercase text, blank text becomes Null"""
    if isinstance(value, str):
        value = value.strip().lower()
        return value if value != "" else None
    return value

silver_df = bronze.copy()   #bronze root table isn't changed

#Only text columns are cleaned
text_cols = [c for c in silver_df.columns
             if not pd.api.types.is_numeric_dtype(silver_df[c])
             and not pd.api.types.is_datetime64_any_dtype(silver_df[c])]
silver_df.columns = [col.strip().replace(' ', '_') for col in silver_df.columns]
silver_df['InvoiceDate'] = silver_df['InvoiceDate'].dt.date


changes = {}
for col in text_cols:
    before = silver_df[col]
    after = before.map(clean_text)
    changes[col] = int(((before.astype(str) != after.astype(str)) & before.notna()).sum())
    silver_df[col] = after

#Flag any row that has at least one missing value
silver_df["MissingData"] = silver_df.isna().any(axis=1)
silver_df["CustomerIdExists"] = silver_df["Customer_ID"].notna()

print("PREVIEW (nothing saved yet)")
print(f"Text columns checked: {text_cols}")
print(f"Cells that WILL change: {sum(changes.values()):,}")
for col, n in changes.items():
    print(f"   {col}: {n:,}")

print(f"Rows flagged as having a missing value: {silver_df['MissingData'].sum():,} of {len(silver_df):,}")
missing = silver_df.drop(columns="MissingData").isna().sum()
print(f"Rows with a Customer ID: {silver_df['CustomerIdExists'].sum():,}")
print(f"Rows without one: {(~silver_df['CustomerIdExists']).sum():,}")
print("Missing values per column:")
print(missing[missing > 0])

PREVIEW (nothing saved yet)
Text columns checked: ['Invoice', 'StockCode', 'Description', 'Country']
Cells that WILL change: 2,280,540
   Invoice: 19,500
   StockCode: 131,515
   Description: 1,062,154
   Country: 1,067,371
Rows flagged as having a missing value: 243,007 of 1,067,371
Rows with a Customer ID: 824,364
Rows without one: 243,007
Missing values per column:
Description      4382
Customer_ID    243007
dtype: int64


In [4]:
answer = input("Apply these changes and save to the silver table? (yes/no): ").strip().lower()

if answer == "yes":
    con.execute("CREATE OR REPLACE TABLE silver AS SELECT * FROM silver_df")
    print(f"Done. Saved {len(silver_df):,} rows to the 'silver' table.")
    print(f"Cells changed: {sum(changes.values()):,}")
    print(f"Rows flagged MissingData: {silver_df['MissingData'].sum():,}")
    
else:
    print("No changes saved.")

con.close()

: 

In [ ]:
# Summary of the table: number of rows, each column's name,
# how many non-missing values it has, and its data type
silver_df.info()

In [ ]:
silver_df.head()